# 03 - Salary Quantile Regression: Training & Evaluation

Objective: train and evaluate the PyTorch quantile salary model using the real Phase 2 embeddings and processed salary targets.

This notebook now expects `models/job_embeddings.npy` and `data/processed/salaries.npy` to exist. It trains on scaled targets internally, then inverse-transforms predictions back to USD for calibration, MAE, residual, and interval-coverage analysis.


In [1]:
import sys
from pathlib import Path

# Ensure project root is importable from either repo root or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from torch.utils.data import DataLoader

from ml.salary_model import (
    SalaryQuantileNet,
    PinballLoss,
    split_data,
    predict_salary,
    QUANTILES,
    SEED,
)

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch {torch.__version__}")
print(f"Quantiles: {QUANTILES}")
print(f"Project root: {PROJECT_ROOT}")


PyTorch 2.10.0
Quantiles: (0.1, 0.25, 0.5, 0.75, 0.9)
Project root: /Users/omerhortig/Documents/Spring '26/Fundamentals of Machine Learning/fml-project


## 1. Data: Real Embeddings and Salary Targets

The notebook uses the artifacts generated by:

```bash
uv run python scripts/preprocess_data.py
uv run python scripts/build_index.py
```

If these files are missing, run those commands before this notebook.


In [2]:
EMBEDDINGS_PATH = PROJECT_ROOT / "models" / "job_embeddings.npy"
SALARIES_PATH = PROJECT_ROOT / "data" / "processed" / "salaries.npy"

missing = [path for path in (EMBEDDINGS_PATH, SALARIES_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing real salary-training artifacts: "
        + ", ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing)
        + ". Run `uv run python scripts/preprocess_data.py` and "
        + "`uv run python scripts/build_index.py` first."
    )

embeddings = np.load(EMBEDDINGS_PATH).astype(np.float32)
salaries = np.load(SALARIES_PATH).astype(np.float32)

if embeddings.ndim != 2:
    raise ValueError(f"Expected 2D embeddings, got shape {embeddings.shape}")
if salaries.ndim != 1:
    raise ValueError(f"Expected 1D salary targets, got shape {salaries.shape}")
if len(embeddings) != len(salaries):
    raise ValueError(f"Row mismatch: {len(embeddings)} embeddings vs {len(salaries)} salaries")

EMB_DIM = embeddings.shape[1]
N_SAMPLES = len(salaries)

print(f"Loaded real data: {N_SAMPLES:,} samples, dim={EMB_DIM}")
print(f"Embedding dtype: {embeddings.dtype}; salary dtype: {salaries.dtype}")
print(f"Salary range: ${salaries.min():,.0f} - ${salaries.max():,.0f}")
print(f"Salary mean:  ${salaries.mean():,.0f}  median: ${np.median(salaries):,.0f}")


Loaded real data: 35,118 samples, dim=384
Embedding dtype: float32; salary dtype: float32
Salary range: $10,000 - $960,000
Salary mean:  $96,688  median: $82,500


## 2. Train / Val / Test Split

In [3]:
train_ds, val_ds, test_ds, scaler = split_data(embeddings, salaries, seed=SEED)
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
print(f"Salary scaler: mean=${scaler.mean:,.0f}, std=${scaler.std:,.0f}")

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)


Train: 28,094  Val: 3,511  Test: 3,513
Salary scaler: mean=$96,780, std=$60,173


## 3. Model Training

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = SalaryQuantileNet(
    embedding_dim=EMB_DIM,
    n_extra_features=0,
    dropout=0.2,
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {param_count:,}")

criterion = PinballLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=4
)

# Real data is larger than the original synthetic notebook; keep training bounded
# so the notebook remains presentation-friendly.
EPOCHS = 60
PATIENCE = 8

best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_train = []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        epoch_train.append(loss.item())

    model.eval()
    epoch_val = []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            epoch_val.append(criterion(model(X_b), y_b).item())

    avg_train = float(np.mean(epoch_train))
    avg_val = float(np.mean(epoch_val))
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    if epoch % 5 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:>3d}  train={avg_train:.4f}  val={avg_val:.4f}  lr={lr:.1e}")

if best_state is None:
    raise RuntimeError("Training did not produce a best checkpoint")

model.load_state_dict(best_state)
model.eval()
print()
print(f"Best val loss: {best_val_loss:.4f}  (stopped at epoch {len(train_losses)})")


Device: cpu
Trainable parameters: 140,933
Epoch   1  train=0.1974  val=0.1671  lr=1.0e-03
Epoch   5  train=0.1406  val=0.1490  lr=1.0e-03
Epoch  10  train=0.1207  val=0.1433  lr=1.0e-03
Epoch  15  train=0.1091  val=0.1406  lr=1.0e-03
Epoch  20  train=0.1005  val=0.1407  lr=1.0e-03
Epoch  25  train=0.0945  val=0.1386  lr=1.0e-03
Epoch  30  train=0.0854  val=0.1391  lr=5.0e-04
Early stopping at epoch 31

Best val loss: 0.1375  (stopped at epoch 31)


### 3.1 Loss Curves

In [5]:
loss_df = pd.DataFrame(
    {
        "epoch": np.arange(1, len(train_losses) + 1),
        "train": train_losses,
        "validation": val_losses,
    }
).melt(id_vars="epoch", var_name="split", value_name="pinball_loss")

fig = px.line(
    loss_df,
    x="epoch",
    y="pinball_loss",
    color="split",
    title="Training & Validation Loss",
    labels={"pinball_loss": "Pinball loss", "epoch": "Epoch"},
)
fig.update_layout(height=430)
fig.show()


[plot rendered: Training & Validation Loss]


## 4. Test-Set Evaluation

Collect predictions on the held-out test set.

In [6]:
model.eval()
all_preds_scaled, all_targets_scaled = [], []

with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        preds = model(X_b).cpu().numpy()
        preds = np.sort(preds, axis=1)  # enforce monotonicity per sample
        all_preds_scaled.append(preds)
        all_targets_scaled.append(y_b.numpy())

all_preds_scaled = np.concatenate(all_preds_scaled, axis=0)
all_targets_scaled = np.concatenate(all_targets_scaled, axis=0)

all_preds = scaler.inverse_transform(all_preds_scaled)
all_targets = scaler.inverse_transform(all_targets_scaled)
all_preds = np.sort(all_preds, axis=1)

print(f"Test samples: {len(all_targets):,}")
print(f"Prediction shape: {all_preds.shape}")
print(f"Predicted q50 range: ${all_preds[:, 2].min():,.0f} - ${all_preds[:, 2].max():,.0f}")


Test samples: 3,513
Prediction shape: (3513, 5)
Predicted q50 range: $24,437 - $362,025


## 5. Calibration Analysis

For each quantile τ, compute the fraction of test samples where the actual salary falls **below** the predicted τ-quantile. Ideal calibration: the fraction equals τ.

In [7]:
calibration = {}
print(f"{'Quantile':>10s}  {'Nominal':>8s}  {'Actual':>8s}  {'Δ':>6s}")
print("-" * 40)

for i, q in enumerate(QUANTILES):
    frac_below = (all_targets < all_preds[:, i]).mean()
    delta = frac_below - q
    calibration[q] = frac_below
    status = "✅" if abs(delta) <= 0.05 else "⚠️"
    print(f"  q{int(q*100):>2d}       {q:.2f}      {frac_below:.3f}   {delta:+.3f}  {status}")

# Acceptance criterion: within ±5 percentage points
max_delta = max(abs(v - k) for k, v in calibration.items())
print(f"\nMax calibration error: {max_delta:.3f}")
print("PASS" if max_delta <= 0.05 else "FAIL (> ±5 pp)")

  Quantile   Nominal    Actual       Δ
----------------------------------------
  q10       0.10      0.128   +0.028  ✅
  q25       0.25      0.254   +0.004  ✅
  q50       0.50      0.482   -0.018  ✅
  q75       0.75      0.705   -0.045  ✅
  q90       0.90      0.840   -0.060  ⚠️

Max calibration error: 0.060
FAIL (> ±5 pp)


In [8]:
calibration_df = pd.DataFrame(
    {
        "nominal": list(calibration.keys()),
        "actual": list(calibration.values()),
    }
)
calibration_df["label"] = calibration_df["nominal"].map(lambda q: f"q{int(q * 100)}")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Perfect calibration",
        line=dict(color="black", dash="dash"),
    )
)
fig.add_trace(
    go.Scatter(
        x=calibration_df["nominal"],
        y=calibration_df["actual"],
        mode="lines+markers+text",
        text=calibration_df["label"],
        textposition="top center",
        name="Model calibration",
    )
)
fig.update_layout(
    title="Quantile Calibration Plot",
    xaxis_title="Nominal quantile",
    yaxis_title="Actual fraction below prediction",
    width=620,
    height=620,
)
fig.update_xaxes(range=[-0.02, 1.02])
fig.update_yaxes(range=[-0.02, 1.02])
fig.show()


[plot rendered: Quantile Calibration Plot]


## 6. Median Prediction MAE

**Acceptance criterion:** MAE < $15,000 on the test set.

In [9]:
median_idx = list(QUANTILES).index(0.50)
median_preds = all_preds[:, median_idx]

residuals = all_targets - median_preds
mae = np.abs(residuals).mean()
rmse = np.sqrt((residuals ** 2).mean())
mape = (np.abs(residuals) / np.clip(all_targets, 1, None)).mean() * 100
baseline_median = np.full_like(all_targets, np.median(all_targets))
baseline_mae = np.abs(all_targets - baseline_median).mean()

print("Median prediction (q50) metrics:")
print(f"  MAE:          ${mae:,.0f}")
print(f"  RMSE:         ${rmse:,.0f}")
print(f"  MAPE:         {mape:.1f}%")
print(f"  Baseline MAE: ${baseline_mae:,.0f}  (test-set median predictor)")
print(f"  Improvement:  {baseline_mae - mae:,.0f} dollars")


Median prediction (q50) metrics:
  MAE:          $22,613
  RMSE:         $42,108
  MAPE:         23.3%
  Baseline MAE: $43,205  (test-set median predictor)
  Improvement:  20,591 dollars


## 7. Residual Analysis

In [10]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Residual Distribution",
        "Predicted vs. Actual",
        "Residuals vs. Predicted",
    ),
)

fig.add_trace(
    go.Histogram(x=residuals, nbinsx=50, name="Residuals", marker_color="#3498db"),
    row=1,
    col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)

lims = [
    float(min(median_preds.min(), all_targets.min())),
    float(max(median_preds.max(), all_targets.max())),
]
fig.add_trace(
    go.Scatter(
        x=median_preds,
        y=all_targets,
        mode="markers",
        name="Predicted vs actual",
        marker=dict(size=4, opacity=0.35, color="#2ecc71"),
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(x=lims, y=lims, mode="lines", name="y = x", line=dict(color="red", dash="dash")),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=median_preds,
        y=residuals,
        mode="markers",
        name="Residual vs predicted",
        marker=dict(size=4, opacity=0.35, color="#9b59b6"),
    ),
    row=1,
    col=3,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=3)

fig.update_xaxes(title_text="Residual ($)", row=1, col=1)
fig.update_xaxes(title_text="Predicted q50 ($)", row=1, col=2)
fig.update_yaxes(title_text="Actual salary ($)", row=1, col=2)
fig.update_xaxes(title_text="Predicted q50 ($)", row=1, col=3)
fig.update_yaxes(title_text="Residual ($)", row=1, col=3)
fig.update_layout(height=460, width=1200, showlegend=False)
fig.show()


[plot rendered: untitled]


## 8. Prediction Interval Fan Chart

Visualise the predicted quantile intervals for a sorted subset of test samples.

In [11]:
# Sort by predicted median for a clean fan chart.
sort_idx = np.argsort(median_preds)
N_show = min(200, len(sort_idx))
step = max(1, len(sort_idx) // N_show)
sel = sort_idx[::step][:N_show]
xs = np.arange(len(sel))

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.concatenate([xs, xs[::-1]]),
        y=np.concatenate([all_preds[sel, 4], all_preds[sel, 0][::-1]]),
        fill="toself",
        fillcolor="rgba(52, 152, 219, 0.15)",
        line=dict(color="rgba(52, 152, 219, 0)"),
        name="q10-q90",
    )
)
fig.add_trace(
    go.Scatter(
        x=np.concatenate([xs, xs[::-1]]),
        y=np.concatenate([all_preds[sel, 3], all_preds[sel, 1][::-1]]),
        fill="toself",
        fillcolor="rgba(52, 152, 219, 0.3)",
        line=dict(color="rgba(52, 152, 219, 0)"),
        name="q25-q75",
    )
)
fig.add_trace(go.Scatter(x=xs, y=all_preds[sel, 2], mode="lines", name="q50 median", line=dict(color="#2c3e50")))
fig.add_trace(go.Scatter(x=xs, y=all_targets[sel], mode="markers", name="Actual", marker=dict(size=5, color="#e74c3c", opacity=0.6)))
fig.update_layout(
    title="Salary Prediction Fan Chart",
    xaxis_title="Test samples sorted by predicted median",
    yaxis_title="Salary ($)",
    height=520,
)
fig.show()


[plot rendered: Salary Prediction Fan Chart]


## 9. Prediction Interval Coverage

In [12]:
# Interval coverage: what fraction of actuals fall within each band?
intervals = [
    ("q10–q90 (80%)", 0, 4, 0.80),
    ("q25–q75 (50%)", 1, 3, 0.50),
]

print(f"{'Interval':>20s}  {'Expected':>9s}  {'Actual':>8s}  {'Δ':>6s}")
print("-" * 50)

for name, lo_idx, hi_idx, expected in intervals:
    in_band = ((all_targets >= all_preds[:, lo_idx]) & 
               (all_targets <= all_preds[:, hi_idx])).mean()
    delta = in_band - expected
    status = "✅" if abs(delta) <= 0.05 else "⚠️"
    print(f"{name:>20s}  {expected:>8.0%}      {in_band:.3f}   {delta:+.3f}  {status}")

            Interval   Expected    Actual       Δ
--------------------------------------------------
       q10–q90 (80%)       80%      0.712   -0.088  ⚠️
       q25–q75 (50%)       50%      0.451   -0.049  ✅


## 10. Per-Quantile MAE

In [13]:
q_labels = [f"q{int(q * 100)}" for q in QUANTILES]
q_maes = [np.abs(all_targets - all_preds[:, i]).mean() for i in range(len(QUANTILES))]
q_mae_df = pd.DataFrame({"quantile": q_labels, "mae": q_maes})

fig = px.bar(
    q_mae_df,
    x="quantile",
    y="mae",
    text=q_mae_df["mae"].map(lambda value: f"${value:,.0f}"),
    title="Mean Absolute Error by Quantile",
    labels={"mae": "MAE ($)", "quantile": "Quantile"},
)
fig.update_traces(textposition="outside")
fig.update_layout(height=430)
fig.show()


[plot rendered: Mean Absolute Error by Quantile]


## 11. Single-Sample Inference Demo

Demonstrate the `predict_salary()` API with a test embedding.

In [14]:
# Pick a deterministic test sample.
sample_idx = min(42, len(test_ds) - 1)
sample_emb = test_ds.X[sample_idx].numpy()[:EMB_DIM]  # strip extra features if any
actual_salary = scaler.inverse_transform(np.array([test_ds.y[sample_idx].item()]))[0]

result = predict_salary(model, sample_emb, scaler=scaler)

print(f"Actual salary: ${actual_salary:,.0f}")
print("Predicted quantiles:")
for k, v in result.items():
    print(f"  {k}: ${v:,.0f}")


Actual salary: $105,000
Predicted quantiles:
  q10: $74,638
  q25: $85,837
  q50: $100,112
  q75: $115,685
  q90: $130,857


## 12. Summary

Real-data salary regression run completed using Phase 2 embeddings and processed salary targets.

Data and training setup:
- Loaded `35,118` real job embeddings from `models/job_embeddings.npy` with dimension `384`.
- Loaded `35,118` annual salary targets from `data/processed/salaries.npy`.
- Salary range was `$10,000` to `$960,000`; mean salary was `$96,688`; median salary was `$82,500`.
- Split sizes: `28,094` train, `3,511` validation, `3,513` test.
- The model trained on scaled salary targets and all reported evaluation metrics were inverse-transformed back to USD.
- Training ran on CPU and stopped early at epoch `31`; best validation pinball loss was `0.1375`.

Evaluation results:
- q50 median prediction MAE: `$22,613`.
- RMSE: `$42,108`.
- MAPE: `23.3%`.
- Test-set median baseline MAE: `$43,205`, so the learned model improves over the baseline by about `$20,591`.
- Calibration was acceptable for q10, q25, q50, and q75 under the +/-5 percentage point rule.
- q90 was slightly under-covered: nominal `0.900`, actual `0.840`, error `-0.060`.
- Interval coverage: q10-q90 expected `80%`, actual `71.2%`; q25-q75 expected `50%`, actual `45.1%`.

Interpretation:
- The real embedding-based salary model is meaningfully better than a median baseline, which is enough to support a working demo and final presentation discussion.
- The upper-tail quantile is still under-calibrated, so the model is too narrow/conservative for high-salary jobs. If time permits, the next tuning pass should address q90 coverage with additional features, loss weighting, longer training, or outlier handling.
